# Saved-output pilot

This is the concrete V5 CPU-first wrapper. Planning is enabled by default; real execution is disabled and requires the separately supplied authorization token. Target labels remain inaccessible until each policy freeze manifest and target-score hash exist.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml
REPO = Path('/kaggle/working/gnn-fraud') if Path('/kaggle').exists() else Path.cwd()
CACHE = Path('/kaggle/input/coregraph-evidence-cache') if Path('/kaggle').exists() else Path(os.environ.get('COREGRAPH_EVIDENCE_CACHE', str(REPO.parent / 'gnn-fraud-local-evidence-cache')))
OUTPUT_ROOT = Path('/kaggle/working/coregraph-v5-pilot') if Path('/kaggle').exists() else REPO / 'results/coregraph_pilot/v5_authorised_run'
CONFIG = REPO / 'configs/coregraph/pilot/saved_output_v5.yaml'
RUNNER = REPO / 'scripts/coregraph/run_saved_output_pilot_v5.py'
EXECUTE = False
PLAN = True
VALIDATE = False
RUN_SYNTHETIC = False
RESUME = True
AUTHORIZATION_TOKEN = ''
CHUNK_ROWS = 50000
MAX_WORKERS = 1
print({'repo':str(REPO),'cache':str(CACHE),'output':str(OUTPUT_ROOT),'execute':EXECUTE})


In [ ]:
required = [RUNNER, CONFIG, CACHE / 'indexes/RB09V3_MEMBER_INDEX.csv']
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing V5 prerequisites: {missing}'
branch = subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
commit_sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert branch == 'codex/coregraph-iclr-buildout-2026', branch
config = yaml.safe_load(CONFIG.read_text())
spec = REPO / config['preregistration_path']
preregistration_sha256 = hashlib.sha256(spec.read_bytes()).hexdigest()
assert preregistration_sha256 == config['preregistration_sha256']
assert len(config['archive_hashes']) == 6
free_bytes = shutil.disk_usage(OUTPUT_ROOT.parent).free
assert free_bytes >= 5 * 1024**3, 'At least 5 GiB free space is required'
print({'branch':branch,'commit_sha':commit_sha,'preregistration_sha256':preregistration_sha256,'archives':6,'free_bytes':free_bytes})


In [ ]:
base = [sys.executable,str(RUNNER),'--config',str(CONFIG),'--evidence-cache',str(CACHE),'--output-root',str(OUTPUT_ROOT),'--chunk-rows',str(CHUNK_ROWS),'--max-workers',str(MAX_WORKERS)]
plan_command = [*base,'--plan']
validate_command = [*base,'--validate-only']
synthetic_command = [sys.executable,str(RUNNER),'--config',str(CONFIG),'--output-root',str(OUTPUT_ROOT.parent/'v5_synthetic_notebook_smoke'),'--synthetic-fixture','--execute','--fail-fast']
real_command = [*base,'--execute','--authorization-token','<EXPLICIT_LATER_AUTHORIZATION>']
print({'plan_command':plan_command,'validate_command':validate_command,'synthetic_command':synthetic_command,'real_command_not_run':real_command,'expected_counts':'6/180/60/540/240'})
if PLAN:
    subprocess.run(plan_command,cwd=REPO,check=True)
if VALIDATE:
    subprocess.run(validate_command,cwd=REPO,check=True)
if RUN_SYNTHETIC:
    subprocess.run(synthetic_command,cwd=REPO,check=True)


In [ ]:
if EXECUTE:
    assert AUTHORIZATION_TOKEN == 'AUTHORIZE_COREGRAPH_V5_PILOT_RUN', 'Missing explicit later authorization'
    command = [*base,'--execute','--authorization-token',AUTHORIZATION_TOKEN]
    if RESUME:
        command.append('--resume')
    subprocess.run(command,cwd=REPO,check=True)
else:
    print('Real saved-output pilot remains disabled; no fit or target metric was run by this cell.')


In [ ]:
plan = OUTPUT_ROOT / 'PILOT_PLAN.csv'
if plan.exists():
    planned = sum(1 for _ in plan.open()) - 1
    assert planned == 240, planned
    completed = len(list(OUTPUT_ROOT.glob('scenarios/*/methods/*/COMPLETE')))
    print({'planned':planned,'completed':completed,'resume':RESUME})
    if EXECUTE and completed == planned:
        subprocess.run([sys.executable,str(RUNNER),'--output-root',str(OUTPUT_ROOT),'--package'],cwd=REPO,check=True)
    elif EXECUTE:
        raise RuntimeError(f'Packaging blocked: {completed}/{planned} coordinates complete')
